In [1]:
import kagglehub
import pathlib
import shutil
import tensorflow as tf
# Download latest version
downloaded_path = pathlib.Path(kagglehub.dataset_download("grassknoted/asl-alphabet"))

Using Colab cache for faster access to the 'asl-alphabet' dataset.


In [2]:
DATA_PATH = pathlib.Path("./data")

if DATA_PATH.exists():
    shutil.rmtree(DATA_PATH)

shutil.copytree(downloaded_path, DATA_PATH)

PosixPath('data')

In [3]:
deletable = pathlib.Path("./data/asl_alphabet_test")

shutil.rmtree(deletable)

In [4]:
!pip install split-folders

In [5]:
import splitfolders

DATASET_PATH = pathlib.Path("./data/asl_alphabet_train/asl_alphabet_train")

SPLITTED = pathlib.Path("./SPLITTED")

splitfolders.ratio(DATASET_PATH, output = SPLITTED, ratio = (.7,.2,.1), group_prefix = None)

Copying files: 87000 files [00:19, 4568.38 files/s]


In [6]:
import os

train_dir = pathlib.Path("./SPLITTED/train")
test_dir = pathlib.Path("./SPLITTED/test")
val_dir = pathlib.Path("./SPLITTED/val")

seed = 42
img_size = (299,299)
batch_size = 16

# Removed disk caching to save space. Relying on prefetching for performance.

train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size = img_size,
    batch_size = batch_size,
    seed = seed,
    shuffle = True
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    image_size = img_size,
    batch_size = batch_size,
    seed = seed
)

test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size = img_size,
    batch_size = batch_size,
    seed = seed
)

train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)
validation_dataset = validation_dataset.prefetch(tf.data.AUTOTUNE)
test_dataset = test_dataset.prefetch(tf.data.AUTOTUNE)

Found 60900 files belonging to 29 classes.
Found 17400 files belonging to 29 classes.
Found 8700 files belonging to 29 classes.


In [7]:
backbone = tf.keras.applications.EfficientNetB4(
    include_top = False,
    weights = "imagenet",
    input_shape = img_size + (3,)
)

backbone.trainable = False

71686520/71686520 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [8]:
classification_heads = tf.keras.models.Sequential(
    [
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(128, activation = "relu"),
        tf.keras.layers.Dense(256, activation = "relu"),
        tf.keras.layers.Dense(29, activation = "softmax")
    ]
)

In [9]:
preprocess_input = tf.keras.applications.efficientnet.preprocess_input

In [10]:
inputs = tf.keras.layers.Input(shape = img_size + (3,))
x = preprocess_input(inputs)
x = backbone(x)
outputs = classification_heads(x)
model = tf.keras.Model(inputs, outputs)

In [11]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

learning_rate_adapter = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=3
)

checkpoint_model = tf.keras.callbacks.ModelCheckpoint(
    filepath="./best.keras",
    monitor="val_loss",
    save_best_only=True
)

checkpoint_weights = tf.keras.callbacks.ModelCheckpoint(
    filepath="./best.weights.h5",
    monitor="val_loss",
    save_best_only=True,
    save_weights_only=True
)

In [12]:
model.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate = 0.0001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

In [13]:
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=100,
    callbacks=[
        early_stopping,
        learning_rate_adapter,
        checkpoint_model,
        checkpoint_weights
    ]
)

Epoch 1/100
3807/3807 ━━━━━━━━━━━━━━━━━━━━ 582s 138ms/step - accuracy: 0.8326 - loss: 0.7237 - val_accuracy: 0.9730 - val_loss: 0.1469 - learning_rate: 1.0000e-04
Epoch 2/100
3807/3807 ━━━━━━━━━━━━━━━━━━━━ 463s 121ms/step - accuracy: 0.9674 - loss: 0.1373 - val_accuracy: 0.9899 - val_loss: 0.0553 - learning_rate: 1.0000e-04
Epoch 3/100
3807/3807 ━━━━━━━━━━━━━━━━━━━━ 473s 124ms/step - accuracy: 0.9815 - loss: 0.0754 - val_accuracy: 0.9947 - val_loss: 0.0302 - learning_rate: 1.0000e-04
Epoch 4/100
3807/3807 ━━━━━━━━━━━━━━━━━━━━ 473s 124ms/step - accuracy: 0.9872 - loss: 0.0505 - val_accuracy: 0.9957 - val_loss: 0.0249 - learning_rate: 1.0000e-04
Epoch 5/100
3807/3807 ━━━━━━━━━━━━━━━━━━━━ 475s 125ms/step - accuracy: 0.9907 - loss: 0.0363 - val_accuracy: 0.9976 - val_loss: 0.0149 - learning_rate: 1.0000e-04
Epoch 6/100
3807/3807 ━━━━━━━━━━━━━━━━━━━━ 478s 126ms/step - accuracy: 0.9921 - loss: 0.0300 - val_accuracy: 0.9987 - val_loss: 0.0089 - learning_rate: 1.0000e-04
Epoch 7/100
3807/3807 

KeyboardInterrupt: 

## 99.92% di validation accuracy all'epoca 10

In [14]:
best_model = tf.keras.models.load_model("./best.keras")

model.load_weights("./best.weights.h5")

In [15]:
results = model.evaluate(test_dataset)
print("Test Loss:", results[0])
print("Test Accuracy:", results[1])

544/544 ━━━━━━━━━━━━━━━━━━━━ 71s 130ms/step - accuracy: 0.9986 - loss: 0.0057
Test Loss: 0.005687388125807047
Test Accuracy: 0.9986206889152527


## 99.86% di accuracy sul test set